# FinRisk — XGBoost Impact Model Training (Memory-Safe)

**Fixed for Colab free tier (12 GB RAM).**

Key changes from v1:
- **Streams GDELT results to JSONL on disk** — never accumulates rows in RAM
- **Processes only 1 file per hour** sampled uniformly — ~8,700 files instead of 35k, ~45 min total
- **Correct GDELT columns** — company names are in col 13/14 (Organizations), not col 7 (theme codes)
- **Chunked label building** — reads JSONL in 50k-row chunks

**Runtime:** ~60–80 min total on Colab free CPU.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q xgboost==2.0.3 yfinance==0.2.40 scipy==1.13.0 tqdm==4.66.2 \
               pyarrow pandas numpy scikit-learn
print('Done.')

## Cell 2 — Configuration

In [ ]:
from pathlib import Path
import json, os

DATA_DIR     = Path('/content/finrisk')
MODELS_DIR   = DATA_DIR / 'models'
GDELT_JSONL  = DATA_DIR / 'gdelt_rows.jsonl'   # streamed to disk, never in RAM
LABELED_OUT  = DATA_DIR / 'labeled_events.parquet'
PERCNT_OUT   = DATA_DIR / 'impact_percentiles.json'
MODEL_OUT    = MODELS_DIR / 'impact_xgb.pkl'
FEATURES_OUT = MODELS_DIR / 'impact_xgb_features.json'

for d in [DATA_DIR, MODELS_DIR]: d.mkdir(parents=True, exist_ok=True)

TICKERS          = ['AAPL','MSFT','NVDA','GOOGL','AMZN','META','TSLA',
                    'JPM','BAC','GS','V','JNJ','UNH','PFE',
                    'XOM','CVX','CAT','BA','WMT','HD','SPY']
UNIVERSE_TICKERS = [t for t in TICKERS if t != 'SPY']

# Company fragments to match — checked against Organizations + URL columns
COMPANY_FILTERS = [
    'apple','microsoft','nvidia','alphabet','google','amazon',
    'meta platforms','facebook','tesla','jpmorgan','jp morgan',
    'bank of america','goldman sachs','visa inc','johnson & johnson',
    'unitedhealth','pfizer','exxon','chevron','caterpillar','boeing',
    'walmart','home depot',
]

TICKER_ALIASES = {
    'AAPL':['apple','apple inc'],
    'MSFT':['microsoft','microsoft corp'],
    'NVDA':['nvidia','nvidia corp'],
    'GOOGL':['alphabet','google','alphabet inc'],
    'AMZN':['amazon','amazon.com','amazon inc'],
    'META':['meta platforms','meta','facebook'],
    'TSLA':['tesla','tesla inc','tesla motors'],
    'JPM':['jpmorgan','jp morgan','jpmorgan chase'],
    'BAC':['bank of america','bofa'],
    'GS':['goldman sachs','goldman'],
    'V':['visa inc','visa'],
    'JNJ':['johnson & johnson','johnson and johnson','j&j'],
    'UNH':['unitedhealth','united health'],
    'PFE':['pfizer'],
    'XOM':['exxon','exxonmobil','exxon mobil'],
    'CVX':['chevron'],
    'CAT':['caterpillar'],
    'BA':['boeing'],
    'WMT':['walmart','wal-mart'],
    'HD':['home depot'],
}

EVENT_TYPES = ['Earnings','M&A','Regulatory','Macro','Geopolitical',
               'Credit','Commodity','Cyber','Leadership','Product']
SECTORS     = ['Technology','Communication Services','Consumer Discretionary',
               'Financials','Health Care','Energy','Industrials','Consumer Staples']
TICKER_SECTOR = {
    'AAPL':'Technology','MSFT':'Technology','NVDA':'Technology',
    'GOOGL':'Communication Services','META':'Communication Services',
    'AMZN':'Consumer Discretionary','TSLA':'Consumer Discretionary','HD':'Consumer Discretionary',
    'JPM':'Financials','BAC':'Financials','GS':'Financials','V':'Financials',
    'JNJ':'Health Care','UNH':'Health Care','PFE':'Health Care',
    'XOM':'Energy','CVX':'Energy',
    'CAT':'Industrials','BA':'Industrials',
    'WMT':'Consumer Staples',
}
BASE_IMPACT = {
    'Earnings':5.0,'M&A':5.5,'Regulatory':6.0,'Macro':6.0,
    'Geopolitical':7.0,'Credit':8.0,'Commodity':5.5,
    'Cyber':6.5,'Leadership':4.5,'Product':4.0,
}

LOOKBACK_DAYS  = 365
# Sample 1 file per hour (365 days × 24 hours = 8,760 files)
# vs downloading every 15-min file (35,040). Same coverage, 4× less data.
SAMPLE_EVERY_N = 4   # take 1 of every 4 files = 1 per hour

print('Config ready.')
print(f'  Will sample ~{35040//SAMPLE_EVERY_N:,} GDELT files (1 per hour)')

## Cell 3 — Fetch GDELT + stream to disk

**Memory-safe: writes rows to JSONL on disk, never accumulates in RAM.**

Samples 1 file per hour (every 4th 15-min file) → ~8,700 files, ~45 min.

**GDELT GKG v2.1 correct column layout:**
- col 1  = DATE (YYYYMMDDHHmmss)
- col 4  = DocumentIdentifier (article URL)  
- col 13 = Organizations ← **company names here**
- col 14 = V2Organizations ← `Apple,1;Microsoft,2;...`
- col 16 = V2Tone ← `overall,positive,negative,...`

In [ ]:
import gzip, io, json, time, gc
from datetime import datetime, timedelta, timezone
import requests
from tqdm.notebook import tqdm

MAX_RETRIES = 2
GDELT_BASE  = 'http://data.gdeltproject.org/gdeltv2'


def _get(url, timeout=30):
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200: return r
            if r.status_code == 404: return None
        except Exception: pass
        if attempt < MAX_RETRIES - 1: time.sleep(2)
    return None


def _row_matches(orgs, v2orgs, url):
    """Check Organizations (col13), V2Orgs (col14), and URL (col4)."""
    v2orgs_names = ' '.join(p.split(',')[0] for p in v2orgs.split(';') if p.strip())
    text = (orgs + ' ' + v2orgs_names + ' ' + url).lower()
    return any(f in text for f in COMPANY_FILTERS)


def _parse_and_write(raw_bytes, fh_out):
    """Parse one GKG file and write matching rows to open file handle. Returns count."""
    written = 0
    try:
        text = raw_bytes.decode('utf-8', errors='replace')
        for line in text.splitlines():
            if not line.strip(): continue
            f = line.split('\t')
            if len(f) < 17: continue
            orgs   = f[13]
            v2orgs = f[14]
            url    = f[4]
            if not _row_matches(orgs, v2orgs, url): continue
            # Parse tone (V2Tone = col 16, comma-separated)
            tone_parts = f[16].split(',')
            try:
                t_overall  = float(tone_parts[0])
                t_positive = float(tone_parts[1]) if len(tone_parts) > 1 else 0.0
                t_negative = float(tone_parts[2]) if len(tone_parts) > 2 else 0.0
            except (ValueError, IndexError):
                t_overall = t_positive = t_negative = 0.0
            row = {
                'DATE': f[1],
                'SourceCommonName': f[3],
                'DocumentIdentifier': url,
                'V2Themes': f[8] if len(f) > 8 else '',
                'Organizations': orgs,
                'V2Organizations': v2orgs,
                'tone_overall': t_overall,
                'tone_positive': t_positive,
                'tone_negative': t_negative,
            }
            fh_out.write(json.dumps(row) + '\n')
            written += 1
    except Exception:
        pass
    return written


def _gdelt_ts(s):
    try: return datetime.strptime(s, '%Y%m%d%H%M%S').replace(tzinfo=timezone.utc)
    except: return None


# ── Build URL list ────────────────────────────────────────────────────────────
end_dt   = datetime.now(tz=timezone.utc)
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)

print('Fetching GDELT master file list ...')
resp = _get('http://data.gdeltproject.org/gdeltv2/masterfilelist.txt', timeout=90)
urls = []
if resp:
    all_gkg = [l.split()[-1] for l in resp.text.splitlines() if l.endswith('.gkg.csv.zip')]
    for url in all_gkg:
        fname = url.split('/')[-1].replace('.gkg.csv.zip','')
        fdt = _gdelt_ts(fname)
        if fdt and start_dt <= fdt <= end_dt:
            urls.append(url)
    urls = sorted(urls)
    print(f'Total files in range: {len(urls):,}')
else:
    # Fallback: construct URLs every 15 min
    print('Master list unavailable — constructing URLs from timestamps')
    cur = start_dt
    while cur <= end_dt:
        m = (cur.minute // 15) * 15
        s = cur.replace(minute=m, second=0, microsecond=0)
        urls.append(f"{GDELT_BASE}/{s.strftime('%Y%m%d%H%M%S')}.gkg.csv.zip")
        cur += timedelta(minutes=15)

# Sample every Nth file to stay within memory and time budget
sampled = urls[::SAMPLE_EVERY_N]
print(f'Sampling 1 of every {SAMPLE_EVERY_N} files → {len(sampled):,} files to process')

# ── Stream to disk ────────────────────────────────────────────────────────────
total_rows = 0
skipped    = 0

with open(GDELT_JSONL, 'w') as fh_out:
    for i, url in enumerate(tqdm(sampled, desc='GDELT files', unit='file')):
        r = _get(url, timeout=45)
        if r is None:
            skipped += 1
            continue
        try:
            with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz:
                csv_bytes = gz.read()
            # Write directly to disk — no accumulation in RAM
            n = _parse_and_write(csv_bytes, fh_out)
            total_rows += n
            del csv_bytes, r  # explicit free
        except Exception:
            skipped += 1
        # Flush every 100 files and collect garbage
        if i % 100 == 0:
            fh_out.flush()
            gc.collect()

print(f'\n✓ Done: {total_rows:,} matching rows written to {GDELT_JSONL}')
print(f'  Files skipped (404/timeout): {skipped:,}')
file_mb = GDELT_JSONL.stat().st_size / 1e6
print(f'  JSONL file size: {file_mb:.1f} MB')

if total_rows == 0:
    raise RuntimeError('Still 0 rows. Run Cell 3b below to debug the column layout.')

## Cell 3b — DEBUG (run if Cell 3 gets 0 rows)

Inspects one actual file to show real column values.

In [ ]:
# Only run this if Cell 3 produced 0 rows
import gzip, io, requests

resp = requests.get('http://data.gdeltproject.org/gdeltv2/masterfilelist.txt', timeout=60)
test_url = [l.split()[-1] for l in resp.text.splitlines() if l.endswith('.gkg.csv.zip')][-1]
print(f'Inspecting: {test_url.split("/")[-1]}')

r = requests.get(test_url, timeout=60)
with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz:
    raw = gz.read().decode('utf-8', errors='replace')

rows = [l for l in raw.splitlines() if l.strip()]
f = rows[0].split('\t')
print(f'Columns in first row: {len(f)}')
print()
for idx, name in {0:'GKGRECORDID',1:'DATE',3:'SourceCommonName',4:'URL',
                  8:'V2Themes',13:'Organizations',14:'V2Organizations',16:'V2Tone'}.items():
    val = f[idx][:100] if idx < len(f) else 'MISSING'
    print(f'  col {idx:2d} [{name}]: {val}')
    print()

# Test filter on 500 rows
hits = []
for line in rows[:500]:
    fs = line.split('\t')
    if len(fs) < 17: continue
    orgs = fs[13]; v2orgs = fs[14]; url = fs[4]
    v2n  = ' '.join(p.split(',')[0] for p in v2orgs.split(';') if p.strip())
    text = (orgs + ' ' + v2n + ' ' + url).lower()
    if any(frag in text for frag in COMPANY_FILTERS):
        hits.append(orgs[:60] or v2n[:60])

print(f'Matches in first 500 rows: {len(hits)}')
for h in hits[:5]: print(f'  {h}')
if not hits:
    # Show what IS in organizations to diagnose
    print('\nSample Organizations values from first 10 rows:')
    for line in rows[:10]:
        fs = line.split('\t')
        if len(fs) > 14:
            print(f'  col13={fs[13][:60]}  |  col14={fs[14][:60]}')

## Cell 4 — Fetch yfinance price data

**Expected time: 2–5 minutes**

In [ ]:
import yfinance as yf, time
from datetime import datetime, timedelta, timezone
from tqdm.notebook import tqdm

daily_start  = (datetime.now(tz=timezone.utc) - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
hourly_start = (datetime.now(tz=timezone.utc) - timedelta(days=59)).strftime('%Y-%m-%d')
fetched = []

for ticker in tqdm(TICKERS, desc='Prices', unit='ticker'):
    t = yf.Ticker(ticker)
    for interval, start in [('1d', daily_start), ('1h', hourly_start)]:
        for attempt in range(3):
            try:
                df = t.history(start=start, interval=interval, auto_adjust=True)
                if not df.empty:
                    df.index = df.index.tz_convert('UTC') if df.index.tz else df.index.tz_localize('UTC')
                    df.index.name = 'datetime'
                    df.to_parquet(DATA_DIR / f'prices_{interval.replace("1d","daily")}_{ticker}.parquet')
                break
            except Exception:
                if attempt < 2: time.sleep(3)
    if ticker not in fetched: fetched.append(ticker)
    time.sleep(0.5)

spy = __import__('pandas').read_parquet(DATA_DIR / 'prices_daily_SPY.parquet')
print(f'\n✓ Prices done. SPY: {len(spy)} bars, {spy.index[0].date()} → {spy.index[-1].date()}')

## Cell 5 — Compute abnormal returns + impact labels

Reads GDELT JSONL in **chunks** — memory safe.

**Expected time: 5–15 minutes**

In [ ]:
import pandas as pd
import numpy as np
import json, gc
from tqdm.notebook import tqdm


def _load_prices(ticker, interval):
    p = DATA_DIR / f'prices_{interval}_{ticker}.parquet'
    if not p.exists(): return None
    df = pd.read_parquet(p)
    if df.index.tz is None: df.index = df.index.tz_localize('UTC')
    else: df.index = df.index.tz_convert('UTC')
    return df['Close'].sort_index()


def _next_after(prices, ts):
    fut = prices[prices.index > ts]
    return (float(fut.iloc[0]), fut.index[0]) if not fut.empty else None


def _price_window(prices, ts, td):
    w = prices[(prices.index > ts) & (prices.index <= ts + td)]
    return float(w.iloc[-1]) if not w.empty else None


def _compute_ar(article_ts, ticker, h_px, d_px, spy_h, spy_d):
    sh = h_px.get(ticker); sd = d_px.get(ticker)
    if sh is None and sd is None: return None
    entry = _next_after(sh, article_ts) if sh is not None else None
    if entry is None and sd is not None: entry = _next_after(sd, article_ts)
    if entry is None: return None
    ep, ets = entry
    assert ets > article_ts  # no look-ahead

    ar_1h = None
    if sh is not None:
        e1h = _price_window(sh, article_ts, pd.Timedelta(hours=1))
        s0 = _next_after(spy_h, article_ts) if spy_h is not None else None
        s1h = _price_window(spy_h, article_ts, pd.Timedelta(hours=1)) if spy_h is not None else None
        if e1h and s0 and s1h:
            sp0, spts = s0
            assert spts > article_ts
            ar_1h = (e1h - ep) / ep - (s1h - sp0) / sp0

    ar_1d = None
    pd_s = sd if sd is not None else sh
    spy_pd = spy_d if spy_d is not None else spy_h
    if pd_s is not None:
        e1d = _price_window(pd_s, article_ts, pd.Timedelta(days=1))
        s0d = _next_after(spy_pd, article_ts) if spy_pd is not None else None
        s1d = _price_window(spy_pd, article_ts, pd.Timedelta(days=1)) if spy_pd is not None else None
        if e1d and s0d and s1d:
            sp0d, sptsd = s0d
            assert sptsd > article_ts
            ar_1d = (e1d - ep) / ep - (s1d - sp0d) / sp0d

    if ar_1h is None and ar_1d is None: return None
    abs_max = max(abs(ar_1h) if ar_1h else 0.0, abs(ar_1d) if ar_1d else 0.0)
    return {'entry_ts': ets, 'ep': ep, 'ar_1h': ar_1h, 'ar_1d': ar_1d, 'abs_ar': abs_max}


def _identify_tickers(orgs, v2orgs, url, v2themes):
    v2n  = ' '.join(p.split(',')[0] for p in v2orgs.split(';') if p.strip())
    text = (orgs + ' ' + v2n + ' ' + url + ' ' + v2themes).lower()
    return [t for t in UNIVERSE_TICKERS
            if any(a and a in text for a in TICKER_ALIASES.get(t, [t.lower()]))]


def _gdelt_ts(s):
    try: return pd.Timestamp(s, tz='UTC').__class__(
             __import__('datetime').datetime.strptime(s,'%Y%m%d%H%M%S'), tz='UTC')
    except: return None


# ── Load prices (small — all fit in RAM) ──────────────────────────────────────
print('Loading price data...')
h_px = {t: _load_prices(t,'1h')     for t in UNIVERSE_TICKERS + ['SPY']}
d_px = {t: _load_prices(t,'daily')  for t in UNIVERSE_TICKERS + ['SPY']}
spy_h = h_px.get('SPY'); spy_d = d_px.get('SPY')
print(f'  SPY 1h: {len(spy_h) if spy_h is not None else "missing"} bars')
print(f'  SPY daily: {len(spy_d) if spy_d is not None else "missing"} bars')
if spy_h is None and spy_d is None:
    raise RuntimeError('SPY prices missing — re-run Cell 4.')

# ── Count GDELT rows first ────────────────────────────────────────────────────
n_gdelt = sum(1 for _ in open(GDELT_JSONL))
print(f'\nProcessing {n_gdelt:,} GDELT rows (streaming from disk)...')

# ── Process JSONL in streaming fashion ───────────────────────────────────────
records = []
look_ahead_errs = 0
CHUNK_FLUSH = 10_000  # write records to list every 10k

with open(GDELT_JSONL) as fh:
    for i, line in enumerate(tqdm(fh, total=n_gdelt, desc='Events')):
        if not line.strip(): continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue

        # Parse timestamp
        try:
            from datetime import datetime, timezone
            article_ts = pd.Timestamp(
                datetime.strptime(str(row['DATE']), '%Y%m%d%H%M%S').replace(tzinfo=timezone.utc)
            )
        except Exception:
            continue

        tickers_found = _identify_tickers(
            row.get('Organizations',''),
            row.get('V2Organizations',''),
            row.get('DocumentIdentifier',''),
            row.get('V2Themes',''),
        )

        for ticker in tickers_found:
            try:
                ar = _compute_ar(article_ts, ticker, h_px, d_px, spy_h, spy_d)
            except AssertionError:
                look_ahead_errs += 1
                continue
            if ar is None: continue

            records.append({
                'article_ts':     article_ts,
                'ticker':         ticker,
                'SourceCommonName': row.get('SourceCommonName',''),
                'DocumentIdentifier': row.get('DocumentIdentifier',''),
                'V2Themes':       row.get('V2Themes',''),
                'tone_overall':   row.get('tone_overall',0.0),
                'tone_positive':  row.get('tone_positive',0.0),
                'tone_negative':  row.get('tone_negative',0.0),
                'entry_ts':       ar['entry_ts'],
                'entry_price':    ar['ep'],
                'ar_1h':          ar['ar_1h'],
                'ar_1d':          ar['ar_1d'],
                'abs_ar_max':     ar['abs_ar'],
            })

        if i % CHUNK_FLUSH == 0:
            gc.collect()

if look_ahead_errs: print(f'  Look-ahead assertions skipped: {look_ahead_errs}')
if not records:
    raise RuntimeError('No events with price matches. Check GDELT time range overlaps price data.')

events_df = pd.DataFrame(records)
print(f'\nProduced {len(events_df):,} event-ticker pairs')

# ── Impact labels from empirical percentiles ──────────────────────────────────
PCTILE_BINS = [(0.,10.,(1,2)),(10.,30.,(3,4)),(30.,60.,(4,6)),(60.,80.,(6,8)),(80.,100.,(8,10))]
abs_ar = events_df['abs_ar_max'].values.astype(float)
p995   = float(np.percentile(abs_ar, 99.5))
clipped = np.clip(abs_ar, 0, p995)
thresholds = [float(np.percentile(clipped, p)) for p,_,_ in PCTILE_BINS[1:]]

def _map_impact(v):
    for i, (_,_,(lo,hi)) in enumerate(PCTILE_BINS):
        th = thresholds[i] if i < len(thresholds) else float('inf')
        if v < th:
            st = thresholds[i-1] if i > 0 else 0.
            bw = th - st
            fr = (v - st) / bw if bw > 0 else 0.5
            return float(lo) + fr * (hi - lo)
    lo,hi = PCTILE_BINS[-1][2]
    return float(lo) + min((v-thresholds[-1])/(thresholds[-1]+1e-9),1.) * (hi-lo)

events_df['impact_label'] = events_df['abs_ar_max'].apply(_map_impact).clip(1.,10.).round(1)
PERCNT_OUT.write_text(json.dumps({'thresholds': thresholds, 'p99_5': p995}, indent=2))
events_df = events_df.sort_values('article_ts').reset_index(drop=True)
events_df.to_parquet(LABELED_OUT, index=False)

print(f'\n✓ Saved {len(events_df):,} labeled events → {LABELED_OUT}')
bins   = [1,2,3,4,5,6,7,8,9,10.1]
labels = ['1-2','2-3','3-4','4-5','5-6','6-7','7-8','8-9','9-10']
counts,_ = np.histogram(events_df['impact_label'].values, bins=bins)
for lbl, cnt in zip(labels, counts):
    print(f'  {lbl}  {cnt:6d}  {"█"*min(cnt//(max(1,len(events_df)//40)),40)}')
del records; gc.collect()

## Cell 6 — Feature engineering + XGBoost training

Time-based 70/15/15 split — no random shuffling.  
**Expected time: 2–5 minutes**

In [ ]:
import pickle
import pandas as pd, numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

df = pd.read_parquet(LABELED_OUT).sort_values('article_ts').reset_index(drop=True)
print(f'Loaded {len(df):,} labeled events')

def _event_type(v2themes):
    t = v2themes.lower()
    if any(k in t for k in ['earn','revenue','eps','quarterly','guidance','profit']): return 'Earnings'
    if any(k in t for k in ['merger','acquisition','takeover','buyout','divest']):    return 'M&A'
    if any(k in t for k in ['regulation','sec_','antitrust','penalty','fine','lawsuit']): return 'Regulatory'
    if any(k in t for k in ['federal_reserve','interest_rate','inflation','gdp','unemployment']): return 'Macro'
    if any(k in t for k in ['sanction','war','tariff','military','geopolit','conflict']): return 'Geopolitical'
    if any(k in t for k in ['credit','debt','downgrade','bankrupt','bond','default']): return 'Credit'
    if any(k in t for k in ['oil','gas','commodity','opec','energy_price','crude']):   return 'Commodity'
    if any(k in t for k in ['cyber','hack','breach','ransomware','malware']):          return 'Cyber'
    if any(k in t for k in ['ceo','cfo','resign','appoint','executive','departure']):  return 'Leadership'
    if any(k in t for k in ['product','launch','recall','patent','drug','approval']):  return 'Product'
    return 'Earnings'

def _is_after_hours(ts):
    try:
        t = pd.Timestamp(ts)
        t = t.tz_localize('UTC') if t.tz is None else t.tz_convert('UTC')
        return t.weekday() >= 5 or not (t.replace(hour=13,minute=30,second=0) <= t < t.replace(hour=20,minute=0,second=0))
    except: return False

# Features
out = pd.DataFrame(index=df.index)
out['sentiment']      = (df['tone_overall']/10.).clip(-1.,1.)
out['abs_sentiment']  = out['sentiment'].abs()
out['tone_positive']  = df['tone_positive'].fillna(0.)
out['tone_negative']  = df['tone_negative'].fillna(0.).abs()
out['source_count']   = 1.
out['is_after_hours'] = df['article_ts'].apply(lambda ts: float(_is_after_hours(ts)) if pd.notna(ts) else 0.)
ds = df.sort_values(['ticker','article_ts']).copy()
ds['prev_ts'] = ds.groupby('ticker')['article_ts'].shift(1)
ds['dsl']     = ((ds['article_ts']-ds['prev_ts']).dt.total_seconds()/86400).fillna(30.).clip(0.,90.)
out['days_since_last'] = ds['dsl'].reindex(df.index).fillna(30.)
event_col = df['V2Themes'].apply(_event_type)
for et in EVENT_TYPES:
    out[f"event_type_{et.lower().replace('&','and').replace(' ','_')}"] = (event_col==et).astype(float)
sec_col = df['ticker'].map(TICKER_SECTOR).fillna('Technology')
for sec in SECTORS:
    out[f"sector_{sec.lower().replace(' ','_')}"] = (sec_col==sec).astype(float)

feat_cols = list(out.columns)
X = out.values.astype(float)
y = df['impact_label'].values.astype(float)

n = len(df)
t0, t1 = int(n*.7), int(n*.85)
X_tr,y_tr = X[:t0],y[:t0]
X_v, y_v  = X[t0:t1],y[t0:t1]
X_te,y_te = X[t1:],y[t1:]
print(f'Split: train={len(X_tr):,} val={len(X_v):,} test={len(X_te):,}')
print(f'Train → {df.iloc[t0]["article_ts"].date()}  |  Test → {df.iloc[t1]["article_ts"].date()}')

model = XGBRegressor(
    objective='reg:squarederror', n_estimators=200, max_depth=4,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, eval_metric='rmse',
    early_stopping_rounds=20, verbosity=0,
)
model.fit(X_tr, y_tr, eval_set=[(X_v,y_v)], verbose=False)
print(f'Best iteration: {model.best_iteration}')

y_pred = np.clip(model.predict(X_te),1.,10.)
et_te  = df['V2Themes'].iloc[t1:].apply(_event_type).values
s_te   = out['sentiment'].iloc[t1:].values
base   = np.clip([BASE_IMPACT.get(e,5.)*(0.5+0.5*abs(s))*0.7*0.9 for e,s in zip(et_te,s_te)],1.,10.)

xgb_mae  = mean_absolute_error(y_te,y_pred)
base_mae = mean_absolute_error(y_te,base)
xgb_rho, _ = spearmanr(y_te,y_pred)
base_rho,_ = spearmanr(y_te,base)

print(f'\n{"="*55}')
print(f'  {"Model":<28} {"MAE":>6} {"Spearman ρ":>12}')
print(f'  {"-"*53}')
print(f'  {"Rule-based baseline":<28} {base_mae:6.3f} {base_rho:+12.3f}')
print(f'  {"XGBoost":<28} {xgb_mae:6.3f} {xgb_rho:+12.3f}')
print(f'{"="*55}')
print(f'  MAE improvement: {(base_mae-xgb_mae)/base_mae*100:+.1f}%')

print('\nTop 10 feature importances:')
for feat, imp in sorted(zip(feat_cols, model.feature_importances_), key=lambda x:-x[1])[:10]:
    print(f'  {feat:<42} {imp:.4f}  {"█"*int(imp*60)}')

# Save
MODELS_DIR.mkdir(parents=True, exist_ok=True)
with open(MODEL_OUT,'wb') as f: pickle.dump(model,f,protocol=pickle.HIGHEST_PROTOCOL)
meta = {
    'feature_columns': feat_cols, 'event_types': EVENT_TYPES, 'sectors': SECTORS,
    'n_train':int(len(X_tr)), 'n_val':int(len(X_v)), 'n_test':int(len(X_te)),
    'best_iteration':int(model.best_iteration),
    'test_mae':float(xgb_mae), 'test_spearman_rho':float(xgb_rho),
    'baseline_mae':float(base_mae), 'baseline_spearman_rho':float(base_rho),
}
FEATURES_OUT.write_text(__import__('json').dumps(meta, indent=2))
print(f'\n✓ Model saved: {MODEL_OUT}\n✓ Features saved: {FEATURES_OUT}')

## Cell 7 — Download model files + print evaluation summary

In [ ]:
from google.colab import files
import json

print('Downloading 3 model files...')
files.download(str(MODEL_OUT))
files.download(str(FEATURES_OUT))
files.download(str(PERCNT_OUT))

print('\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  EVALUATION RESULTS (for your slide)')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
m = json.loads(FEATURES_OUT.read_text())
print(f'  Training samples:   {m["n_train"]:,}')
print(f'  Test samples:       {m["n_test"]:,}')
print(f'  XGBoost MAE:        {m["test_mae"]:.3f}')
print(f'  XGBoost Spearman ρ: {m["test_spearman_rho"]:+.3f}')
print(f'  Baseline MAE:       {m["baseline_mae"]:.3f}')
print(f'  Baseline Spearman:  {m["baseline_spearman_rho"]:+.3f}')
print(f'  Best iteration:     {m["best_iteration"]}')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print()
print('After downloading:')
print('  1. Place all 3 files in finrisk/data/models/')
print('  2. Add MODEL_IMPACT=xgboost to finrisk/.env')
print('  3. Restart the API — XGBoost loads automatically')